# Consumer Price Index (CPI) - Data Cleaning 

### Data Info:
* **Indicator:** `Індекс споживчих цін (CPI)`
* **Source:** National Bank of Ukraine: https://bank.gov.ua/ua/statistic/macro-indicators
* **Raw File:** `data_raw/CPI.xlsx`
* **Output:** `data_processed/CPI_clean.xlsx`  

### Cleaning Notes:
- wide format: rows = indicators, columns = dates
- ?

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [2]:
file_path = Path("../../data_raw/CPI.xlsx")
output_path = Path("../../data_processed/CPI_clean.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

sheet_name = "Sheet1"

df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
df_raw

,0,1,2,3,4,5,6,7,8,9,...,113,114,115,116,117,118,119,120,121,122
0,NaN,2015-12-01 00:00:00,2016-01-01 00:00:00,2016-02-01 00:00:00,2016-03-01 00:00:00,2016-04-01 00:00:00,2016-05-01 00:00:00,2016-06-01 00:00:00,2016-07-01 00:00:00,2016-08-01 00:00:00,...,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,2026-01-01 00:00:00
1,Індекс споживчих цін,43.3,40.3,32.7,20.9,9.8,7.5,6.9,7.9,8.4,...,15.1,15.9,14.3,14.1,13.2,11.9,10.9,9.3,8,7.4
2,Продукти харчування та безалкогольні напої,41.5,38.1,29.7,11.4,6.9,3.9,3.2,4.4,5.1,...,19.8,22.1,23.2,22.6,20.5,17.4,15.6,12.1,10.2,9.7
3,"Алкогольні напої, тютюнові вироби",22.7,15.1,9.8,7.7,8.8,9.9,10.2,11,12.4,...,17.4,18.3,18.3,19.2,19.2,19,18.6,18.5,17.3,16.8
4,Одяг і взуття,35,31.5,26.1,23,19.6,18.4,18.2,19.7,17.1,...,-4.4,-4.8,-4.5,-5.4,-5.7,-5.3,-6,-6.1,-6.1,-5.4
5,"Житло, вода, електроенергія, газ та інші види ...",103,103.4,100.8,104.3,17,13.3,12.1,13.8,14.3,...,20.1,20.1,2.4,2.4,2.4,2.3,2.4,2.3,2.3,2.2
6,"Предмети домашнього вжитку, побутова техніка т...",36,32.2,24.3,13.4,8.4,6.4,6.3,6,5.6,...,1.7,2.4,2.5,2.2,2,1.6,1.7,1.7,1.4,1.3
7,Охорона здоров’я,29.1,26.8,23.3,13.5,9.9,9,9.5,9,8.9,...,13.5,13.1,12.9,12.3,11.8,11.1,9.9,8,6,4.7
8,Транспорт,20.4,18.5,1.8,-4.2,2.5,5.1,5.2,5.4,6,...,6.9,5.8,6.2,7.1,6.5,5.8,5.4,5.9,6.1,5.9
9,Зв’язок,7,6.7,5.8,4.2,4.7,4.2,4.1,3.9,3.7,...,18.5,18.5,17.3,15.1,17.3,17.1,16.4,16.3,12,7.1


In [3]:
# set correct header

raw_headers = ["indicator"] + df_raw.iloc[0, 1:].tolist()

df = df_raw.iloc[1:].copy()
df.columns = raw_headers
df = df.reset_index(drop=True)

df.head(1)

,indicator,2015-12-01 00:00:00,2016-01-01 00:00:00,2016-02-01 00:00:00,2016-03-01 00:00:00,2016-04-01 00:00:00,2016-05-01 00:00:00,2016-06-01 00:00:00,2016-07-01 00:00:00,2016-08-01 00:00:00,...,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00,2025-07-01 00:00:00,2025-08-01 00:00:00,2025-09-01 00:00:00,2025-10-01 00:00:00,2025-11-01 00:00:00,2025-12-01 00:00:00,2026-01-01 00:00:00
0,Індекс споживчих цін,43.3,40.3,32.7,20.9,9.8,7.5,6.9,7.9,8.4,...,15.1,15.9,14.3,14.1,13.2,11.9,10.9,9.3,8,7.4


In [4]:
# clean column names

def clean_date_label(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    x = re.sub(r"\*+", "", x)
    x = re.sub(r"\s*\(.*?\)", "", x)
    x = x.strip()
    
    return x

cleaned_columns = ["indicator"]
for col in df.columns[1:]:
    cleaned_columns.append(clean_date_label(col))

df.columns = cleaned_columns

print("number of parsed date columns:", sum(pd.notna(df.columns[1:])))
print("number of unparsed date columns:", sum(pd.isna(df.columns[1:])))

number of parsed date columns: 122
number of unparsed date columns: 0


In [5]:
# filter needed indicator

target_indicator = "Індекс споживчих цін"

df["indicator"] = (
    df["indicator"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

mask = df["indicator"].str.contains(target_indicator, case=False, na=False)
df_filtered = df.loc[mask].copy()
df_filtered["indicator"] = "CPI"

print("matched rows:", len(df_filtered))
df_filtered[["indicator"]].head()

matched rows: 1


,indicator
0,CPI


In [6]:
# transform to long format (melt)

valid_date_cols = [col for col in df_filtered.columns[1:] if pd.notna(col)]

df_long = df_filtered.melt(
    id_vars="indicator",
    value_vars=valid_date_cols,
    var_name="date",
    value_name="value"
)
df_long = df_long[["date", "value", "indicator"]]
df_long["date"] = pd.to_datetime(df_long["date"], format='ISO8601')
df_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       122 non-null    datetime64[ns]
 1   value      122 non-null    object        
 2   indicator  122 non-null    object        
dtypes: datetime64[ns](1), object(2)
memory usage: 3.0+ KB


In [7]:
# convert values to numeric

df_long["value"] = (
    df_long["value"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.replace(r"[^\d\.\-]", "", regex=True)
    .replace("", np.nan)
)

df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")

df_long.head()

,date,value,indicator
0,2015-12-01,43.3,CPI
1,2016-01-01,40.3,CPI
2,2016-02-01,32.7,CPI
3,2016-03-01,20.9,CPI
4,2016-04-01,9.8,CPI


In [8]:
# validate 

df_clean = df_long[["date", "value", "indicator"]].copy()

print(df_clean.dtypes)
print("\nmissing dates:", df_clean["date"].isna().sum())
print("missing values:", df_clean["value"].isna().sum())
print("unique indicators:", df_clean["indicator"].nunique())

date         datetime64[ns]
value               float64
indicator            object
dtype: object

missing dates: 0
missing values: 0
unique indicators: 1


In [9]:
# save

df_clean["date"] = pd.to_datetime(df_clean["date"]).dt.date

df_clean.to_excel(output_path, index=False)
print(f"saved to: {output_path}")

saved to: ..\..\data_processed\CPI_clean.xlsx
